In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import torch
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence
import pandas as pd
import sentencepiece as spm

class AGNewsDataset(Dataset):
  def __init__(self, csv_path, model_path, max_len=128):
    self.data = pd.read_csv(csv_path)
    self.sp = spm.SentencePieceProcessor()
    self.sp.load(model_path)
    self.max_len = max_len

    self.cls_id = self.sp.piece_to_id('[CLS]')
    self.sep_id = self.sp.piece_to_id('[SEP]')
    self.pad_id = self.sp.pad_id()

  def __len__(self):
    return len(self.data)

  def __getitem__(self, idx):
    row = self.data.iloc[idx]
    text = str(row['full text'])
    label = int(row['Class Index'])
    token_ids = self.sp.encode_as_ids(text)
    token_ids = token_ids[:(self.max_len - 2)]
    token_ids = [self.cls_id] + token_ids + [self.sep_id]

    return (torch.tensor(token_ids), torch.tensor(label))

In [ ]:
def collate_fn(batch, pad_id=0):
  token_ids, label = zip(*batch)
  padded_token_ids = pad_sequence(token_ids, batch_first=True, padding_value=pad_id)
  attention_mask = (padded_token_ids != pad_id).long()
  return padded_token_ids, torch.stack(label), attention_mask